# Pregunta 5 (profesor): PDFs de dry spells en 4 cuencas

## Qué pide
En **4 cuencas** distintas, calcular la **PDF** (función de densidad de probabilidad) de la **duración de dry spells** para:

| Fuente | Periodo |
|--------|---------|
| **CR2MET** | Histórico 1980–2014 |
| **ALADIN histórico** | 1980–2014 |
| **ALADIN futuro** | 2040–2074 (SSP5-8.5, ventana de 35 años comparable) |

Repetir bajo los **3 criterios de umbral** del punto 4:

| Criterio | CR2MET | ALADIN |
|----------|--------|--------|
| **i. Global** | `< 1 mm/día` | `< τ*_dominio` (5.285 mm; calibrado en histórico) |
| **ii. Mismo umbral** | `< 1 mm/día` | `< 1 mm/día` |
| **iii. Local** | `< 1 mm/día` | `< τ*(x,y)` local (calibrado en histórico, fijo para futuro) |

## Cuencas (máscaras rectangulares aproximadas en grilla ALADIN)
1. **Loa** (norte árido)
2. **Maipo** (centro, RM)
3. **Maule** (centro-sur)
4. **Biobío** (sur)

## Metodología
- Grilla de referencia: **ALADIN CHP12**; CR2MET interpolado a esa grilla.
- **Pool regional:** se agrupan todas las duraciones de dry spells de los píxeles dentro de cada cuenca (cada spell conserva su **año de inicio**).
- **PDF:** histograma con bins logarítmicos (`density=True`).
- τ* global y τ* local se calculan solo con datos **históricos** (1980–2014) y se aplican igual al ALADIN futuro.
- **ALADIN hist vs futuro:** bootstrap **por años** (re-muestreo con reemplazo de años dentro de cada periodo) para estimar IC95 de Δmediana, Δmedia y Δp99; significativo si el IC95 no incluye 0.

In [3]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
from IPython.display import display
from shapely.geometry import Point
from shapely.ops import unary_union
from shapely.prepared import prep

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

# =====================================================================
# CONFIGURACION
# =====================================================================
HIST_START = '1980-01-01'
HIST_END = '2014-12-31'
FUT_START = '2040-01-01'
FUT_END = '2074-12-31'
FUTURE_SCENARIO = 'ssp585'

TAU_CR2MET_REF = 1.0
TAU_ALADIN_DOMINIO = 5.285
BISECTION_TAU_MAX = 15.0
BISECTION_TOL = 1e-4

PDF_BINS = 25
MIN_DURATION = 1
BOOTSTRAP_ITER = 1000
RANDOM_SEED = 42
BOOTSTRAP_METRICS = {
    'mediana': np.median,
    'media': np.mean,
    'p99': lambda x: np.percentile(x, 99),
}

# Cuencas: (lon_min, lon_max, lat_min, lat_max)
BASIN_SPECS = {
    'Loa':      {'bounds': (-69.8, -68.2, -24.0, -21.5), 'color': 'goldenrod'},
    'Maipo':    {'bounds': (-71.8, -69.8, -34.5, -33.2), 'color': 'firebrick'},
    'Maule':    {'bounds': (-72.8, -71.0, -36.5, -34.8), 'color': 'darkorange'},
    'Biobio':   {'bounds': (-73.8, -71.5, -38.5, -36.5), 'color': 'forestgreen'},
}

DATASETS_STYLE = {
    'CR2MET (hist)':       {'color': 'black',  'ls': '-',  'lw': 2.2},
    'ALADIN (hist)':       {'color': 'steelblue', 'ls': '-', 'lw': 2.0},
    'ALADIN (futuro)':     {'color': 'crimson', 'ls': '--', 'lw': 2.0},
}

SCENARIOS = {
    'i_global': {
        'title': 'Criterio i — τ* global (5.285 mm en ALADIN)',
        'thresh_cr2': TAU_CR2MET_REF,
        'thresh_ala_key': 'global',
    },
    'ii_mismo': {
        'title': 'Criterio ii — mismo umbral (1 mm)',
        'thresh_cr2': TAU_CR2MET_REF,
        'thresh_ala_key': 'same',
    },
    'iii_local': {
        'title': 'Criterio iii — τ* local por pixel',
        'thresh_cr2': TAU_CR2MET_REF,
        'thresh_ala_key': 'local',
    },
}


def load_chile_geometry():
    reader = shpreader.Reader(
        shpreader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
    )
    geoms = [r.geometry for r in reader.records()
             if r.attributes.get('NAME') == 'Chile' or r.attributes.get('ADMIN') == 'Chile']
    return unary_union(geoms)


def build_chile_mask_on_aladin_grid(lat2d, lon2d, geometry):
    prepared = prep(geometry)
    flat = np.fromiter(
        (prepared.contains(Point(float(x), float(y))) or geometry.touches(Point(float(x), float(y)))
         for y, x in zip(lat2d.ravel(), lon2d.ravel())),
        dtype=bool, count=lat2d.size,
    )
    return flat.reshape(lat2d.shape)


def build_basin_mask(lat2d, lon2d, chile_mask_bool, bounds):
    lon_min, lon_max, lat_min, lat_max = bounds
    in_box = (
        (lon2d >= lon_min) & (lon2d <= lon_max) &
        (lat2d >= lat_min) & (lat2d <= lat_max) & chile_mask_bool
    )
    return in_box


def open_aladin_period(start, end, scenario=None):
    if scenario is None:
        files = sorted(__import__('pathlib').Path('pr1').glob('pr_CHP12_*_historical_*.nc'))
    else:
        files = sorted(__import__('pathlib').Path('pr1').glob(f'pr_CHP12_*_{scenario}_*.nc'))
    ds = xr.open_mfdataset([str(p) for p in files], use_cftime=True, chunks={'time': 365})
    return ds['pr'].sel(time=slice(start, end)) * 86400.0


def open_cr2met_period(start, end):
    ds = xr.open_mfdataset('./pr/CR2MET_pr_v2.5_day_*.nc', chunks={'time': 365})
    return ds['pr'].sel(time=slice(start, end))


def regrid_cr2met_to_aladin(pr_cr2met, template):
    return pr_cr2met.interp(lat=template['lat'], lon=template['lon'], method='linear')


print('1/4: Cargando datos historicos y futuros...')
pr_ala_hist = open_aladin_period(HIST_START, HIST_END)
pr_ala_fut = open_aladin_period(FUT_START, FUT_END, scenario=FUTURE_SCENARIO)
pr_cr2_hist = regrid_cr2met_to_aladin(open_cr2met_period(HIST_START, HIST_END), pr_ala_hist)

chile_geom = load_chile_geometry()
lat2d = pr_ala_hist['lat'].values
lon2d = pr_ala_hist['lon'].values
chile_mask_bool = build_chile_mask_on_aladin_grid(lat2d, lon2d, chile_geom)
chile_mask = xr.DataArray(
    chile_mask_bool,
    coords={'y': pr_ala_hist['y'], 'x': pr_ala_hist['x'], 'lat': pr_ala_hist['lat'], 'lon': pr_ala_hist['lon']},
    dims=['y', 'x'], name='chile_mask',
)

basin_masks = {}
for name, spec in BASIN_SPECS.items():
    bm = build_basin_mask(lat2d, lon2d, chile_mask_bool, spec['bounds'])
    basin_masks[name] = xr.DataArray(
        bm, coords=chile_mask.coords, dims=['y', 'x'], name=f'mask_{name}',
    )
    print(f'  {name}: {int(bm.sum())} pixeles')

print(f'Periodo hist: {HIST_START[:4]}-{HIST_END[:4]} | futuro: {FUT_START[:4]}-{FUT_END[:4]} ({FUTURE_SCENARIO})')

1/4: Cargando datos historicos y futuros...
  Loa: 298 pixeles
  Maipo: 154 pixeles
  Maule: 171 pixeles
  Biobio: 243 pixeles
Periodo hist: 1980-2014 | futuro: 2040-2074 (ssp585)


In [4]:
# =====================================================================
# UMBRALES (global y local) + EXTRACCION DE DRY SPELLS
# =====================================================================
def run_lengths_1d(bool_series):
    x = np.asarray(bool_series, dtype=np.bool_)
    if not np.any(x):
        return np.array([], dtype=np.int16)
    padded = np.r_[False, x, False]
    dx = np.diff(padded.astype(np.int8))
    starts, ends = np.where(dx == 1)[0], np.where(dx == -1)[0]
    return (ends - starts).astype(np.int16)


def wet_fraction_1d(pr_series, threshold):
    x = np.asarray(pr_series, dtype=float)
    x = x[np.isfinite(x)]
    return float((x >= threshold).mean()) if x.size else np.nan


def find_threshold_for_target_1d(pr_series, target_fraction, tau_max=BISECTION_TAU_MAX, tol=BISECTION_TOL):
    if not np.isfinite(target_fraction):
        return np.nan
    f0, fmax = wet_fraction_1d(pr_series, 0.0), wet_fraction_1d(pr_series, tau_max)
    if target_fraction > f0 + tol or target_fraction < fmax - tol:
        return np.nan
    lo, hi = 0.0, tau_max
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        fmid = wet_fraction_1d(pr_series, mid)
        if abs(fmid - target_fraction) <= tol:
            return mid
        lo, hi = (mid, hi) if fmid > target_fraction else (lo, mid)
    return 0.5 * (lo + hi)


def pixelwise_local_aladin_threshold(pr_cr2, pr_aladin, tau_cr2, mask_da):
    f_target = (pr_cr2 >= tau_cr2).mean(dim='time').where(mask_da)
    stacked = xr.Dataset({
        'pr_ala': pr_aladin.stack(cell=('y', 'x')),
        'f_target': f_target.stack(cell=('y', 'x')),
    }).compute()
    n_cells = stacked.sizes['cell']
    tau_star = np.full(n_cells, np.nan, dtype=np.float32)
    for idx in range(n_cells):
        ft = float(stacked['f_target'].values[idx])
        if np.isfinite(ft):
            tau_star[idx] = find_threshold_for_target_1d(stacked['pr_ala'].values[:, idx], ft)
    tau_da = xr.DataArray(tau_star, coords={'cell': stacked['cell']}, dims=['cell']).unstack('cell')
    return tau_da.assign_coords(lat=mask_da['lat'], lon=mask_da['lon'])


def time_to_year(t):
    if hasattr(t, 'year'):
        return int(t.year)
    return int(pd.Timestamp(t).year)


def extract_basin_spell_records(pr, dry_threshold, basin_mask):
    # Pool regional: cada spell guarda duracion y ano de inicio (para bootstrap por anos).
    is_dry = (pr < dry_threshold).where(basin_mask)
    stacked = is_dry.stack(cell=('y', 'x')).transpose('time', 'cell').compute()
    times = stacked['time'].values  # cftime compatible (ALADIN)
    vals = stacked.values
    records = []
    for idx in range(vals.shape[1]):
        col = vals[:, idx]
        if not np.any(col):
            continue
        x = np.asarray(col, dtype=np.bool_)
        padded = np.r_[False, x, False]
        dx = np.diff(padded.astype(np.int8))
        starts, ends = np.where(dx == 1)[0], np.where(dx == -1)[0]
        for s, e in zip(starts, ends):
            duration = int(e - s)
            if duration > 0:
                records.append({'start_year': time_to_year(times[s]), 'duration': duration})
    return pd.DataFrame(records)


def spell_durations_from_records(spell_df):
    if spell_df.empty:
        return np.array([], dtype=np.int32)
    return spell_df['duration'].to_numpy(dtype=np.int32)


def pdf_from_durations(durations, n_bins=PDF_BINS, min_duration=MIN_DURATION):
    d = np.asarray(durations, dtype=float)
    d = d[np.isfinite(d) & (d >= min_duration)]
    if d.size < 50:
        return None, None
    lo = max(float(np.min(d)), min_duration)
    hi = float(np.max(d))
    if hi <= lo:
        return None, None
    bins = np.logspace(np.log10(lo), np.log10(hi), n_bins + 1)
    counts, edges = np.histogram(d, bins=bins, density=True)
    centers = np.sqrt(edges[:-1] * edges[1:])
    return centers, counts


def summarize_durations(durations):
    d = np.asarray(durations, dtype=float)
    d = d[np.isfinite(d) & (d >= 1)]
    if d.size == 0:
        return {}
    return {
        'n_spells': int(d.size),
        'mean': float(np.mean(d)),
        'median': float(np.median(d)),
        'p90': float(np.percentile(d, 90)),
        'p99': float(np.percentile(d, 99)),
        'max': float(np.max(d)),
    }


print('2/4: Calculando tau* local (historico)...')
tau_local_map = pixelwise_local_aladin_threshold(
    pr_cr2_hist.where(chile_mask), pr_ala_hist.where(chile_mask), TAU_CR2MET_REF, chile_mask,
)

THRESH_ALA = {
    'global': TAU_ALADIN_DOMINIO,
    'same': TAU_CR2MET_REF,
    'local': tau_local_map,
}

print('3/4: Extrayendo duraciones por cuenca / criterio / dataset...')
records = []
pdf_cache = {}
spell_df_cache = {}

for scen_key, scen in SCENARIOS.items():
    tc = scen['thresh_cr2']
    ta = THRESH_ALA[scen['thresh_ala_key']]
    for basin_name, basin_mask in basin_masks.items():
        spell_dfs = {
            'CR2MET (hist)': extract_basin_spell_records(
                pr_cr2_hist.where(basin_mask), tc, basin_mask,
            ),
            'ALADIN (hist)': extract_basin_spell_records(
                pr_ala_hist.where(basin_mask), ta, basin_mask,
            ),
            'ALADIN (futuro)': extract_basin_spell_records(
                pr_ala_fut.where(basin_mask), ta, basin_mask,
            ),
        }
        spell_df_cache[(scen_key, basin_name)] = spell_dfs
        durs = {k: spell_durations_from_records(df) for k, df in spell_dfs.items()}
        pdf_cache[(scen_key, basin_name)] = durs
        for ds_name, arr in durs.items():
            s = summarize_durations(arr)
            if s:
                records.append({
                    'criterio': scen_key,
                    'cuenca': basin_name,
                    'dataset': ds_name,
                    **s,
                })

summary_df = pd.DataFrame(records)
display(summary_df.round(2))
print('Listo. Combinaciones PDF:', len(pdf_cache))

2/4: Calculando tau* local (historico)...
3/4: Extrayendo duraciones por cuenca / criterio / dataset...


AttributeError: 'numpy.datetime64' object has no attribute 'year'

In [ ]:
# =====================================================================
# GRAFICOS PDF + BOOTSTRAP POR ANOS (ALADIN hist vs futuro)
# =====================================================================
def bootstrap_delta_by_year(spell_df_hist, spell_df_fut, metric_fn, n_iter=BOOTSTRAP_ITER, seed=RANDOM_SEED):
    """IC95 de (metric_fut - metric_hist) re-muestreando anos con reemplazo."""
    grouped_hist = {
        int(year): grp['duration'].to_numpy()
        for year, grp in spell_df_hist.groupby('start_year')
        if not grp.empty
    }
    grouped_fut = {
        int(year): grp['duration'].to_numpy()
        for year, grp in spell_df_fut.groupby('start_year')
        if not grp.empty
    }
    years_hist = np.array(sorted(grouped_hist))
    years_fut = np.array(sorted(grouped_fut))
    if years_hist.size == 0 or years_fut.size == 0:
        return np.array([])

    rng = np.random.default_rng(seed)
    deltas = []
    for _ in range(n_iter):
        draw_hist = rng.choice(years_hist, size=years_hist.size, replace=True)
        draw_fut = rng.choice(years_fut, size=years_fut.size, replace=True)
        sample_hist = np.concatenate([grouped_hist[y] for y in draw_hist if grouped_hist[y].size > 0])
        sample_fut = np.concatenate([grouped_fut[y] for y in draw_fut if grouped_fut[y].size > 0])
        if sample_hist.size == 0 or sample_fut.size == 0:
            continue
        deltas.append(float(metric_fn(sample_fut) - metric_fn(sample_hist)))
    return np.asarray(deltas)


def plot_basin_pdf(basin_name, scen_key, scen_title, durs_dict):
    fig, ax = plt.subplots(figsize=(10, 6))
    basin_color = BASIN_SPECS[basin_name]['color']

    for ds_name, style in DATASETS_STYLE.items():
        centers, counts = pdf_from_durations(durs_dict[ds_name])
        if centers is None:
            continue
        ax.plot(centers, counts, label=ds_name, **style)

    ax.set_xscale('log')
    ax.set_xlabel('Duracion de dry spell (dias)')
    ax.set_ylabel('PDF p(t)')
    ax.set_title(
        f'PDF dry spells — Cuenca {basin_name}\n{scen_title}\n'
        f'Hist: {HIST_START[:4]}-{HIST_END[:4]} | Futuro ALADIN: {FUT_START[:4]}-{FUT_END[:4]} ({FUTURE_SCENARIO})',
        fontweight='bold',
    )
    ax.legend()
    plt.tight_layout()
    plt.show()


print('4/4: Generando PDFs (12 figuras: 4 cuencas x 3 criterios)...')
for scen_key, scen in SCENARIOS.items():
    print(f"--- {scen['title']} ---")
    for basin_name in BASIN_SPECS:
        plot_basin_pdf(basin_name, scen_key, scen['title'], pdf_cache[(scen_key, basin_name)])

# Bootstrap por anos: ALADIN hist vs futuro por cuenca, criterio y metrica
boot_rows = []
for scen_key in SCENARIOS:
    for basin_name in BASIN_SPECS:
        df_hist = spell_df_cache[(scen_key, basin_name)]['ALADIN (hist)']
        df_fut = spell_df_cache[(scen_key, basin_name)]['ALADIN (futuro)']
        if df_hist.empty or df_fut.empty:
            continue
        for metric_name, metric_fn in BOOTSTRAP_METRICS.items():
            obs_hist = float(metric_fn(df_hist['duration'].to_numpy()))
            obs_fut = float(metric_fn(df_fut['duration'].to_numpy()))
            delta_obs = obs_fut - obs_hist
            boot = bootstrap_delta_by_year(df_hist, df_fut, metric_fn)
            if boot.size == 0:
                continue
            ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
            boot_rows.append({
                'criterio': scen_key,
                'cuenca': basin_name,
                'metrica': metric_name,
                'hist': obs_hist,
                'futuro': obs_fut,
                'delta_obs': delta_obs,
                'delta_boot_mediana': float(np.median(boot)),
                'IC95_inferior': float(ci_low),
                'IC95_superior': float(ci_high),
                'sig_aumento': bool(ci_low > 0),
                'sig_disminucion': bool(ci_high < 0),
            })

bootstrap_df = pd.DataFrame(boot_rows)
print('\nBootstrap por anos — ALADIN hist vs futuro (IC95 de la diferencia):')
display(bootstrap_df.round(3))
print('sig_aumento / sig_disminucion: True si el IC95 de delta no incluye 0.')